[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C44_Adversarial_Security_Course/04_membership_privacy/04_membership_privacy.ipynb)

# 04 · 成员推断与隐私（用 numpy）

目标：在玩具模型上从零实现 **loss 阈值 MIA**、**shadow model MIA**，用 **TPR@low FPR** 正确评价，量化 **过拟合-隐私关联**，并用 **正则/加噪** 防御（DP 直觉）。

> **防御视角**：在合成数据上演示，目的是学会审计自己模型的隐私泄露、验证防御有效。

路线：过拟合靶子 → loss/置信度 MIA → shadow model MIA → TPR@FPR 评价 → 过拟合-隐私曲线 → 加噪防御 → ✏️ 练习 → 📖 答案 → 🧪 LiRA 直觉胶囊。

## 1 · 故意过拟合的靶子

MIA 吃的是**过拟合**。用**高维、少样本**让玩具模型过拟合（训练损失远低于测试），制造成员/非成员的可分性。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_data(n, d=30, seed=0):
    r = np.random.default_rng(seed)
    X = r.standard_normal((n, d))
    w_true = r.standard_normal(d)
    y = (X @ w_true + 0.5*r.standard_normal(n) > 0).astype(int)  # 带噪声标签→可过拟合
    return X, y

def sigmoid(z): return 1/(1+np.exp(-z))
class LogReg:
    def __init__(s,d): s.w=np.zeros(d); s.b=0.0
    def prob(s,X): return sigmoid(X@s.w+s.b)
    def predict(s,X): return (s.prob(X)>0.5).astype(int)
    def fit(s,X,y,lr=0.3,epochs=2000,l2=0.0):
        n=len(y)
        for _ in range(epochs):
            p=s.prob(X); s.w-=lr*(X.T@(p-y)/n+l2*s.w); s.b-=lr*np.mean(p-y)
        return s
def bce(model, X, y):
    p = np.clip(model.prob(X), 1e-9, 1-1e-9)
    return -(y*np.log(p)+(1-y)*np.log(1-p))      # 逐样本损失
def acc(m,X,y): return float(np.mean(m.predict(X)==y))

Xtr, ytr = make_data(120, d=30, seed=1)          # 成员（训练集）
Xout, yout = make_data(120, d=30, seed=2)        # 非成员（同分布，未训练）
model = LogReg(30).fit(Xtr, ytr, l2=0.0)
print(f'训练精度={acc(model,Xtr,ytr):.3f}  测试精度={acc(model,Xout,yout):.3f}')
gap = acc(model,Xtr,ytr) - acc(model,Xout,yout)
print(f'泛化间隙={gap:.3f} (大→过拟合→易被 MIA)')
assert gap > 0.1, '应当过拟合（训练远好于测试）'
print('✅ 过拟合靶子就绪')

## 2 · loss 阈值 MIA + 置信度变体

成员损失低、非成员损失高。设阈值即可判别。用 **AUC** 先粗看攻击强度（下一节再讲为什么 AUC 不够）。

In [ ]:
def auc_score(scores, labels):
    '''labels: 1=成员,0=非成员; scores: 越大越像成员。Mann-Whitney AUC。'''
    pos = scores[labels==1]; neg = scores[labels==0]
    # P(随机成员分数 > 随机非成员分数)
    comp = (pos[:,None] > neg[None,:]).mean() + 0.5*(pos[:,None]==neg[None,:]).mean()
    return float(comp)

loss_in  = bce(model, Xtr, ytr)
loss_out = bce(model, Xout, yout)
print(f'成员平均损失={loss_in.mean():.3f}  非成员={loss_out.mean():.3f}')
# 攻击分数：用 -loss（损失越低越像成员→分数越高）
scores = np.concatenate([-loss_in, -loss_out])
labels = np.concatenate([np.ones(len(loss_in)), np.zeros(len(loss_out))])
auc_loss = auc_score(scores, labels)
print(f'loss-MIA 的 AUC={auc_loss:.3f}')
assert auc_loss > 0.6, '过拟合模型上 loss-MIA 应明显优于随机(0.5)'
# 置信度变体：用对正确类的置信度
conf_in  = np.where(ytr==1, model.prob(Xtr), 1-model.prob(Xtr))
conf_out = np.where(yout==1, model.prob(Xout), 1-model.prob(Xout))
auc_conf = auc_score(np.concatenate([conf_in,conf_out]), labels)
print(f'置信度-MIA 的 AUC={auc_conf:.3f} (与 loss-MIA 同源)')
print('✅ loss/置信度阈值即可推断成员（过拟合让两分布分开）')

## 3 · Shadow model MIA

训多个 shadow 模型（已知各自成员），收集成员/非成员输出做 (特征, in/out) 数据，训一个攻击分类器。攻击特征用 [损失, 置信度] 等。

In [ ]:
def make_attack_data(n_shadow=20, n_each=120, d=30):
    feats = []; lbls = []
    for s in range(n_shadow):
        Xs, ys = make_data(n_each, d=d, seed=100+s)         # shadow 训练集(成员)
        Xo, yo = make_data(n_each, d=d, seed=500+s)         # 非成员
        sm = LogReg(d).fit(Xs, ys, l2=0.0)
        for X_, y_, m_ in [(Xs,ys,1), (Xo,yo,0)]:
            loss = bce(sm, X_, y_)
            conf = np.where(y_==1, sm.prob(X_), 1-sm.prob(X_))
            feats.append(np.stack([loss, conf], 1)); lbls.append(np.full(len(y_), m_))
    return np.vstack(feats), np.concatenate(lbls)

Af, Al = make_attack_data()
attack = LogReg(2).fit(Af, Al.astype(int), l2=1e-3, epochs=1000)
# 用攻击分类器攻目标模型
def target_feats(model, X, y):
    loss = bce(model, X, y); conf = np.where(y==1, model.prob(X), 1-model.prob(X))
    return np.stack([loss, conf], 1)
feat_in  = target_feats(model, Xtr, ytr)
feat_out = target_feats(model, Xout, yout)
sc_shadow = np.concatenate([attack.prob(feat_in), attack.prob(feat_out)])
auc_shadow = auc_score(sc_shadow, labels)
print(f'shadow-model MIA 的 AUC={auc_shadow:.3f}')
assert auc_shadow > 0.6, 'shadow MIA 应优于随机'
print('✅ shadow model：用仿制模型造 in/out 监督信号，训攻击分类器')

## 4 · 正确评价：TPR @ low FPR（而非平均/AUC）

隐私威胁在**最坏情况**：能否在**极低误报**下确信指认少数成员。算 TPR@FPR=1% —— ROC 左端，比 AUC 更能反映真实风险。

In [ ]:
def tpr_at_fpr(scores, labels, target_fpr=0.01):
    '''在给定 FPR 下的 TPR：阈值取使非成员误报率 <= target_fpr 的最严处。'''
    neg = np.sort(scores[labels==0])[::-1]           # 非成员分数降序
    k = max(1, int(target_fpr*len(neg)))
    thr = neg[k-1]                                   # 让约 target_fpr 比例非成员越过
    tpr = float(np.mean(scores[labels==1] >= thr))
    return tpr, thr

tpr_loss, _   = tpr_at_fpr(np.concatenate([-loss_in,-loss_out]), labels, 0.01)
tpr_shadow, _ = tpr_at_fpr(sc_shadow, labels, 0.01)
print(f'loss-MIA   : AUC={auc_loss:.3f}  TPR@1%FPR={tpr_loss:.3f}')
print(f'shadow-MIA : AUC={auc_shadow:.3f}  TPR@1%FPR={tpr_shadow:.3f}')
print('解读：AUC 看“平均区分力”，TPR@low FPR 看“能否确信指认少数人”——后者才是隐私风险')
assert 0 <= tpr_loss <= 1
print('✅ 用低 FPR 端评价 MIA（Carlini 2022 的核心方法论）')

## 5 · 过拟合-隐私关联曲线

核心因果：**泛化间隙越大 → MIA 越成功**。训练不同正则强度的模型，画 (泛化间隙, MIA AUC) 正相关。

In [ ]:
def gap_and_mia(l2):
    m = LogReg(30).fit(Xtr, ytr, l2=l2)
    gap = acc(m,Xtr,ytr) - acc(m,Xout,yout)
    s = np.concatenate([-bce(m,Xtr,ytr), -bce(m,Xout,yout)])
    return gap, auc_score(s, labels)

print(f"{'L2正则':>8}{'泛化间隙':>10}{'MIA AUC':>10}")
rows = []
for l2 in [0.0, 0.01, 0.05, 0.2, 1.0]:
    g, a = gap_and_mia(l2); rows.append((g,a))
    print(f'{l2:>8}{g:>10.3f}{a:>10.3f}')
# 间隙最大者 MIA 也应最强
gaps = [r[0] for r in rows]; aucs = [r[1] for r in rows]
assert aucs[np.argmax(gaps)] >= aucs[np.argmin(gaps)] - 0.02, '间隙大→MIA强（大体正相关）'
print('✅ 过拟合-隐私关联：泛化间隙越大，MIA 越成功 → 用攻击度量隐私')

## 6 · 防御：降过拟合 / 输出加噪（DP 直觉）

强正则降低过拟合 → 缓解 MIA；对输出**加噪**模糊单样本信号（DP-SGD 加噪的玩具直觉）。验证防御后 MIA 变弱。

In [ ]:
# 防御1：强正则
g0, a0 = gap_and_mia(0.0)
g1, a1 = gap_and_mia(0.5)
print(f'无正则: 间隙={g0:.3f} MIA AUC={a0:.3f}')
print(f'强正则: 间隙={g1:.3f} MIA AUC={a1:.3f}')
assert a1 <= a0 + 0.02, '强正则应缓解 MIA'

# 防御2：对损失信号加噪（攻击者拿到的输出被噪声掩盖）
def mia_auc_with_noise(sigma, seed=0):
    r = np.random.default_rng(seed)
    s = np.concatenate([-loss_in, -loss_out]) + r.normal(0, sigma, len(labels))
    return auc_score(s, labels)
for sig in [0.0, 0.5, 1.5, 3.0]:
    print(f'  输出加噪 σ={sig:<4} → MIA AUC={mia_auc_with_noise(sig):.3f}')
assert mia_auc_with_noise(3.0) < mia_auc_with_noise(0.0), '加噪应降低 MIA AUC'
print('✅ 防御：降过拟合 + 加噪都削弱 MIA（DP-SGD 正是裁剪+加噪的有原则版本）')

---
## ✏️ 练习区

### ✏️ 练习 1：从零实现 loss 阈值 MIA 的准确率

实现 `loss_mia_acc(tau)`：用损失阈值 `tau`（损失<tau 判成员）对成员+非成员判别，返回**平衡准确率**。

In [ ]:
def loss_mia_acc(tau):
    # TODO: 成员判 (loss_in<tau) 应为1，非成员判 (loss_out<tau) 应为0；返回两者准确率均值
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 在合理阈值上应优于 0.5
best = max(loss_mia_acc(t) for t in np.linspace(0.1, 2.0, 40))
print(f'最佳阈值下的平衡准确率={best:.3f}')
assert best > 0.55, 'loss 阈值 MIA 应优于随机'
print('✅ 练习 1 通过')

### ✏️ 练习 2：手写 AUC（秩统计）

实现 `auc_rank(scores, labels)`：用**秩**计算 AUC（Mann-Whitney），不用 O(n²) 两两比较。提示：AUC =（成员的秩和 − 成员内部对数）/（成员数×非成员数）。

In [ ]:
def auc_rank(scores, labels):
    # TODO: 用 scipy 风格的秩。可用 np.argsort 两次得到秩；按公式算 AUC
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
s = np.concatenate([-loss_in, -loss_out])
a_rank = auc_rank(s, labels)
a_ref = auc_score(s, labels)
print(f'秩法 AUC={a_rank:.4f}  参考={a_ref:.4f}')
assert abs(a_rank - a_ref) < 1e-6, '秩法应与定义一致'
print('✅ 练习 2 通过')

### ✏️ 练习 3：TPR@low FPR 暴露 AUC 掩盖的风险

构造两组分数：A 组 AUC 高但低 FPR 端弱；B 组 AUC 相近但有少数成员分数极高（低 FPR 端强）。实现 `compare_tpr()` 返回两组的 `(AUC, TPR@1%FPR)`，验证「AUC 相近但 TPR@low FPR 差距大」。

In [ ]:
def compare_tpr():
    r = np.random.default_rng(0)
    lab = np.concatenate([np.ones(500), np.zeros(500)])
    # A: 成员/非成员整体略有偏移（AUC 中等，低FPR端弱）
    A = np.concatenate([r.normal(0.4,1,500), r.normal(0,1,500)])
    # B: 大多数重叠，但少数成员分数极高（低FPR端强）
    B = np.concatenate([r.normal(0,1,500), r.normal(0,1,500)]); B[:30] += 6
    # TODO: 返回 {'A':(auc,tpr@1%), 'B':(auc,tpr@1%)}，用 auc_score 与 tpr_at_fpr
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
res = compare_tpr()
print('A:', res['A'], ' B:', res['B'])
# B 的低 FPR 端 TPR 应明显高于 A（即便 AUC 未必更高）
assert res['B'][1] > res['A'][1], 'B 在低 FPR 端应更危险'
print('✅ 练习 3 通过：低 FPR 端揭示了 AUC 掩盖的“确信指认少数人”风险')

### ✏️ 练习 4：正则强度对 MIA 的缓解

实现 `mia_vs_reg(l2_list)`：返回各正则强度下的 MIA AUC 列表，验证**强正则的 AUC ≤ 无正则**。

In [ ]:
def mia_vs_reg(l2_list):
    # TODO: 对每个 l2，用 gap_and_mia 取 AUC，返回列表
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
aucs = mia_vs_reg([0.0, 0.1, 1.0])
print('MIA AUC vs 正则:', [f'{a:.3f}' for a in aucs])
assert aucs[-1] <= aucs[0] + 0.02, '强正则应缓解 MIA'
print('✅ 练习 4 通过')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def loss_mia_acc(tau):
    acc_in  = np.mean(loss_in  < tau)        # 成员应判成员(1)
    acc_out = np.mean(loss_out >= tau)       # 非成员应判非成员(0)
    return float((acc_in + acc_out) / 2)

In [ ]:
# 练习 2 参考答案：秩法 AUC
def auc_rank(scores, labels):
    order = np.argsort(scores)
    ranks = np.empty(len(scores)); ranks[order] = np.arange(1, len(scores)+1)
    n_pos = int((labels==1).sum()); n_neg = int((labels==0).sum())
    rank_sum_pos = ranks[labels==1].sum()
    return float((rank_sum_pos - n_pos*(n_pos+1)/2) / (n_pos*n_neg))

In [ ]:
# 练习 3 参考答案
def compare_tpr():
    r = np.random.default_rng(0)
    lab = np.concatenate([np.ones(500), np.zeros(500)])
    A = np.concatenate([r.normal(0.4,1,500), r.normal(0,1,500)])
    B = np.concatenate([r.normal(0,1,500), r.normal(0,1,500)]); B[:30] += 6
    out = {}
    for name, s in [('A',A), ('B',B)]:
        out[name] = (auc_score(s,lab), tpr_at_fpr(s,lab,0.01)[0])
    return out

In [ ]:
# 练习 4 参考答案
def mia_vs_reg(l2_list):
    return [gap_and_mia(l2)[1] for l2 in l2_list]

---
## 🧪 真实数据胶囊：LiRA 的 per-sample 校准直觉

复现 Carlini 2022 LiRA 的核心（玩具版）：**全局阈值** 对所有样本一刀切，忽略了「不同样本本来该有多大损失」；
**per-sample 校准**（用 shadow 估计每个样本作为成员/非成员时的损失分布，做似然比）在低 FPR 端更强。

这里玩具化为：把每个目标样本的损失，用「该样本在 shadow 非成员模型上的平均损失」做**校准（减去基线）**，看校准后 MIA 是否更强。

### 胶囊练习：用 shadow 基线校准损失

实现 `calibrate`：给目标样本的损失 `loss` 与该样本在 shadow 非成员上的基线损失 `base`，返回校准分数 `base - loss`（高 = 比预期更像成员）。理解 LiRA 为何要 per-sample。

In [ ]:
def calibrate(loss, base):
    '''per-sample 校准：减去该样本的“非成员预期损失”基线。返回越大越像成员的分数。'''
    # TODO: 返回 base - loss
    raise NotImplementedError

In [ ]:
# —— 胶囊自测 ——（先做 TODO）
# 为每个目标样本估计“非成员基线损失”：在不含该样本的 shadow 模型上的平均损失
def shadow_baseline(Xq, yq, n_shadow=15, d=30):
    losses = np.zeros((len(yq), n_shadow))
    for s in range(n_shadow):
        Xs, ys = make_data(120, d=d, seed=900+s)
        sm = LogReg(d).fit(Xs, ys)
        losses[:, s] = bce(sm, Xq, yq)
    return losses.mean(1)

base_in  = shadow_baseline(Xtr, ytr)
base_out = shadow_baseline(Xout, yout)
# 未校准：直接 -loss ;  校准：base - loss
uncal = np.concatenate([-bce(model,Xtr,ytr), -bce(model,Xout,yout)])
cal   = np.concatenate([calibrate(bce(model,Xtr,ytr), base_in),
                        calibrate(bce(model,Xout,yout), base_out)])
tpr_uncal,_ = tpr_at_fpr(uncal, labels, 0.05)
tpr_cal,_   = tpr_at_fpr(cal,   labels, 0.05)
print(f'未校准 TPR@5%FPR={tpr_uncal:.3f}  |  per-sample校准 TPR@5%FPR={tpr_cal:.3f}')
assert tpr_cal >= tpr_uncal - 1e-9, 'per-sample 校准在低 FPR 端应不弱于全局阈值'
print('✅ 胶囊通过：考虑“样本本该多大损失”的 per-sample 校准，是 LiRA 强于全局阈值的关键')

In [ ]:
# 📖 胶囊参考答案
def calibrate(loss, base):
    return base - loss

### 小结
- **MIA** 判“某样本是否被训练过”，是最基础的隐私攻击，也是**隐私审计**的标准探针。
- **loss/置信度阈值** 简单有效；**shadow model** 造 in/out 监督训攻击分类器，更强。
- 评价要看 **TPR@low FPR**（能否确信指认少数人），而非平均/AUC（Carlini 2022）。
- 根源是 **记忆/过拟合**：泛化间隙越大 MIA 越强；防御=降过拟合 + **差分隐私(DP-SGD：裁剪+加噪)**。

下一站：**模块 05 · Prompt 注入与供应链安全** —— 攻击面扩张到 LLM 应用与整条交付链。